In [1]:
%load_ext autoreload
%autoreload 2
import re
import numpy as np
import read_file, ploting, data_handling, data_mod
import pandas as pd
import os


In [3]:
# import os
# work_path = os.getcwd().split("\\")[:-1]
# work_path = "\\".join(work_path)
# os.chdir(work_path)
# os.getcwd()

## Ajout gullies

In [5]:
# gully_path = r'Inputs/modif/gully_post_9_1m.asc'
# luse_path = r'Inputs/modif/luse_post_9_1m.asc'
# meta_gully, data_gully = read_file.read_asc_file(gully_path, ignore_first_line=False)
# meta_luse, data_luse = read_file.read_asc_file(luse_path, ignore_first_line=True)
# # ploting.create_plotly_map(data_gully, meta_gully, "luse_post_9_1m_gully.asc", color="reds")

In [6]:

# data_luse_gully = data_luse.copy()
# data_luse_gully[(data_gully==1) & (~np.isnan(data_luse_gully))] = 26
# np.nanmax(data_luse_gully)
# data_handling.write_asc_file("luse_post_9_1m_gully.asc", meta_gully, data_luse_gully, "luse_post_9_1m_gully.asc")
# ploting.create_plotly_map(data_luse_gully, meta_gully, "luse_post_9_1m_gully.asc", color="rainbow")

## Modification drainage input

In [19]:
def get_inflows_nodes(file_path: str, ) -> pd.DataFrame:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist.")
    with open(file_path, 'r',) as file:
        nodes = []
        lire_nodes = False
        for line in file:
            if line.strip() == '[INFLOWS]':
                lire_nodes = True
                continue
            if lire_nodes and line.strip().startswith('['):
                break  # Arrêter à la prochaine section
            if lire_nodes and line.strip() and not line.startswith(';;'):
                nodes.append(line.strip())
    if not nodes:
        raise ValueError("No node coordinates found in the [COORDINATES] section.")
    # Convertion en DataFrame
    # df_inflows = pd.DataFrame([node.split() for node in nodes],)
    # df_inflows = df_inflows.apply(pd.to_numeric)
    # df_inflows["Node"] = df_inflows["Node"].astype(int)
    return [int(node.split()[0]) for node in nodes]

list_gully_nodes = get_inflows_nodes(r"input_init/reseau_post_10_2m.txt")
print(list_gully_nodes)

[153, 154, 167, 168, 181, 183, 184, 190, 195, 202, 203, 216, 269, 326, 336, 340, 356]


In [20]:
path_reseau = r"input_init/reseau_post_10_2m.txt"
luse_path = r"input_init/luse_post_10_2m_gully.asc"
meta_luse, data_luse = read_file.read_asc_file(luse_path, ignore_first_line=True)
cellsize = meta_luse["cellsize"]
nodes_coord = read_file.get_nodes_coord(path_reseau, )
gully_coord = nodes_coord.copy()
gully_coord = gully_coord.loc[gully_coord["Node"].isin(list_gully_nodes), ]
# print(gully_coord)
nrows, ncols = data_luse.shape
gully_coord["col"] = ((gully_coord.X - meta_luse["xllcorner"]) / cellsize).astype(int)
gully_coord["row"] = nrows - 1 - np.floor((gully_coord.Y - meta_luse["yllcorner"]) / cellsize).astype(int)

print(gully_coord)


    Node           X            Y  col  row
0    153  620869.103  6876378.456   39  100
1    154  620853.270  6876375.871   31  101
5    167  620850.564  6876407.595   30   85
6    168  620962.409  6876424.071   86   77
9    181  620970.150  6876441.473   90   68
10   183  620959.935  6876459.854   85   59
11   184  620843.445  6876446.568   26   65
12   190  620957.936  6876476.535   84   50
13   195  620939.794  6876497.077   75   40
17   202  620878.370  6876512.370   44   33
18   203  620898.511  6876515.009   54   31
19   216  620800.720  6876548.926    5   14
20   269  620835.845  6876476.291   23   51
21   326  620967.293  6876399.594   88   89
25   336  620839.732  6876463.059   25   57
26   340  620923.710  6876506.117   67   36
27   356  620804.032  6876537.508    7   20


In [21]:
nrows, ncols = data_luse.shape
grid_gully = np.zeros((nrows, ncols))
for id, row, col in zip(gully_coord.Node, gully_coord.row, gully_coord.col):
    grid_gully[row][col] = id

ploting.create_plotly_map(grid_gully, meta_luse)

In [22]:
gully_coord.sort_values(["row", "col"], inplace=True, ignore_index=True )
gully_coord

,Node,X,Y,col,row
0,216,620800.720,6876548.926,5,14
1,356,620804.032,6876537.508,7,20
2,203,620898.511,6876515.009,54,31
3,202,620878.370,6876512.370,44,33
4,340,620923.710,6876506.117,67,36
5,195,620939.794,6876497.077,75,40
6,190,620957.936,6876476.535,84,50
7,269,620835.845,6876476.291,23,51
8,336,620839.732,6876463.059,25,57
9,183,620959.935,6876459.854,85,59


In [24]:
with open("input_init/reseau_post_10_2m_cor.inp", "r") as f:
    data= f.readlines()

index_add = data.index("[INFLOWS]\n") + 4

text_to_add = [f"{node}    FLOW    {i}    FLOW    1.0    1    \n" for node, i in 
               zip(gully_coord.Node, range(1,len(gully_coord.Node)+1))]

data[index_add:index_add] = text_to_add
# text_to_add
# a = [1,2,3,4,5,6,7,8]
# a[2:2] = ["a","b","c"]
# a
data[index_add-4:index_add+ 20]

with open("input_init/reseau_post_10_2m_cor.inp", "w") as f:
    f.write("".join(data))
          

[1, 2, 'a', 'b', 'c', 3, 1000, 1001, 1002, 4, 5, 6, 7, 8]

## Correction MNT

In [ ]:
meta, elev_init = read_file.read_asc_file(file_path="Inputs/modif/elev_9_rgealti_1m.asc",)
ploting.create_plotly_map(elev_init, meta, "Elev correction 0.25m", color="rainbow")

#### Les pixels dont l'écart entre l'élévation la moyenne des évélations des pixels adjacents est supérieur à 0.3m prennent la valeur de la moyenne

In [ ]:
pb, elev_cor, nb_iter = data_mod.cor_elev_pb(elev_init, diff_max=0.25, nb_max=0, ignore_center=True)
data_handling.write_asc_file("Inputs/modif/elev_9_rgealti_cor25_1m.asc", metadata=meta, grid=elev_cor, title="Elevation lissee avec moyenne sur (3,3) ecart de 0.25m resolution 1m")

c:\Users\elie.tisseur\PY3.13.2\Lib\site-packages\scipy\ndimage\_filters.py:2420: RuntimeWarning:

Mean of empty slice



Iter  1    11  pixels aberrants
Iter  2    7  pixels aberrants
Iter  3    6  pixels aberrants
Iter  4    3  pixels aberrants
Iter  5    2  pixels aberrants
Iter  6    1  pixels aberrants
Iter  7    0  pixels aberrants
Inputs/modif/elev_9_rgealti_cor25_1m.asc as been created


In [ ]:
ploting.create_plotly_map(elev_cor, meta, "Elev correction 0.25m", color="rainbow")

## Modification de la grille d'élévation
#### La grille d'élévation est modifiée en fonction du landuse

In [37]:
dict_luse = read_file.create_dict_luse("input_init/Surface_input_post_10.txt")
print(dict_luse)
import csv
land_config = []
with open("input_init/land_config.csv", "r") as f:
    data = csv.DictReader(f)
    i=0
    for row in data:
        i += 1
        land_config.append(row)



param_mod_elev = {name: elev for name, elev in zip([land_config[i]["name"] for i in range(len(land_config))], [float(land_config[i]["modif_elev"]) for i in range(len(land_config))])}
param_mod_elev


{1: {'id': 1, 'conduction': 1.9e-06, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Forest', 'manning': 0.8, 'intercept_depth': 7.62}, 2: {'id': 2, 'conduction': 1.9e-06, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Grass', 'manning': 0.8, 'intercept_depth': 3.81}, 3: {'id': 3, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Road', 'manning': 0.012, 'intercept_depth': 1.9}, 4: {'id': 4, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Impervious Surface', 'manning': 0.012, 'intercept_depth': 1.9}, 5: {'id': 5, 'conduction': 10.0, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Toit', 'manning': 0.012, 'intercept_depth': 1.9}, 6: {'id': 6, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Terre', 'manning': 0.8, 'intercept_depth': 1.9}, 7: {'id': 7, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Revetementimper', 'manning': 0.012, 'intercept_depth': 1.9}, 8: {'id': 8, 'conduction':

{'Forest': 0.0,
 'Grass': 0.0,
 'Road': -0.05,
 'Impervious Surface': 0.0,
 'Toit': 6.0,
 'Terre': 0.0,
 'Revetementimper': 0.0,
 'Copeau1': -0.1,
 'Copeau2': -0.1,
 'Copeau3': -0.1,
 'Copeau4': -0.1,
 'Noue1': -0.05,
 'Noue2': -0.05,
 'Noue3': -0.05,
 'Noue4': -0.05,
 'Noue5': -0.05,
 'Noue6': -0.05,
 'Ilotvegetal': -0.05,
 'Toitc1': 6.0,
 'Toitc3': 6.0,
 'Toitc4': 6.0,
 'Toitn1': 6.0,
 'Toitn2': 6.0,
 'Toitn3': 6.0,
 'Toitn6': 6.0,
 'Arbre1': 0.0,
 'Arbre2': 0.0,
 'Gully': -0.3}

In [38]:
print(dict_luse)

{1: {'id': 1, 'conduction': 1.9e-06, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Forest', 'manning': 0.8, 'intercept_depth': 7.62}, 2: {'id': 2, 'conduction': 1.9e-06, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Grass', 'manning': 0.8, 'intercept_depth': 3.81}, 3: {'id': 3, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Road', 'manning': 0.012, 'intercept_depth': 1.9}, 4: {'id': 4, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Impervious Surface', 'manning': 0.012, 'intercept_depth': 1.9}, 5: {'id': 5, 'conduction': 10.0, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Toit', 'manning': 0.012, 'intercept_depth': 1.9}, 6: {'id': 6, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Terre', 'manning': 0.8, 'intercept_depth': 1.9}, 7: {'id': 7, 'conduction': 1e-10, 'cap_suction': 0.01, 'moisture_def': 0.1, 'name': 'Revetementimper', 'manning': 0.012, 'intercept_depth': 1.9}, 8: {'id': 8, 'conduction':

In [39]:
for sol, mod in param_mod_elev.items():
    print(f"Modification de l'élévation pour {sol}: {mod} m")

Modification de l'élévation pour Forest: 0.0 m
Modification de l'élévation pour Grass: 0.0 m
Modification de l'élévation pour Road: -0.05 m
Modification de l'élévation pour Impervious Surface: 0.0 m
Modification de l'élévation pour Toit: 6.0 m
Modification de l'élévation pour Terre: 0.0 m
Modification de l'élévation pour Revetementimper: 0.0 m
Modification de l'élévation pour Copeau1: -0.1 m
Modification de l'élévation pour Copeau2: -0.1 m
Modification de l'élévation pour Copeau3: -0.1 m
Modification de l'élévation pour Copeau4: -0.1 m
Modification de l'élévation pour Noue1: -0.05 m
Modification de l'élévation pour Noue2: -0.05 m
Modification de l'élévation pour Noue3: -0.05 m
Modification de l'élévation pour Noue4: -0.05 m
Modification de l'élévation pour Noue5: -0.05 m
Modification de l'élévation pour Noue6: -0.05 m
Modification de l'élévation pour Ilotvegetal: -0.05 m
Modification de l'élévation pour Toitc1: 6.0 m
Modification de l'élévation pour Toitc3: 6.0 m
Modification de l'élév

In [40]:
meta_luse, data_luse = read_file.read_asc_file(r"input_init/luse_post_10_2m_gully.asc", ignore_first_line=True)
meta_elev, data_elev = read_file.read_asc_file(r"input_init/elev_post_10_2m.asc", ignore_first_line=True)
elev_grid_mod = data_elev.copy()
for sol, mod in param_mod_elev.items():
    mask_grid = data_luse == [dict_luse[i]["id"] for i in dict_luse.keys() if dict_luse[i]["name"] == sol][0]
    elev_grid_mod = elev_grid_mod + mod * mask_grid
data_handling.write_asc_file("input_init/elev_post_10_2m_mod.asc", metadata=meta_elev, grid=elev_grid_mod, title="Elevation modifiée")
ploting.create_plotly_map(elev_grid_mod, meta_elev, "Elevation modifiée", color="rainbow")

input_init/elev_post_10_2m_mod.asc as been created


In [ ]:
ploting.create_plotly_map_soil(data_luse_gully,meta_luse, dict_luse, "Land use post 9 1m",palette=['#3a7535', "#FFB428", "#5A5A5A", "#00B1BE", "#13b604", "#ff1707", '#7a807a',
                                                                                                   '#3a7535', "#FFB428", "#5A5A5A", "#00B1BE", "#13b604", "#ff1707", '#7a807a',
                                                                                                   '#3a7535', "#FFB428", "#5A5A5A", "#00B1BE", "#13b604", "#ff1707", '#7a807a',
                                                                                                   '#3a7535', "#FFB428", "#5A5A5A", "#00B1BE", "#13b604", "#ff1707", '#7a807a'])

In [ ]:
from io import StringIO
pd.read_xml("info.xml")

,name,code_sol,modif_elev,sfn,infilt_inf,h_subverse,to_SWMM,to_TREX
0,Forest,1,0.00,False,True,False,False,False
1,Grass,2,0.00,False,True,False,False,False
2,Road,3,-0.05,False,False,False,False,False
3,Impervious Surface,4,0.00,False,False,False,False,False
4,Housefree,5,5.00,False,False,False,False,Noue_70


In [ ]:
ploting.create_plotly_map(elev_init, meta ,"Elev init", grids_hover=[elev_init], info_hover=["elev"], color="blues")

In [ ]:
# for i in range (0,2):
#     elev_pb , elev_cor = cor_elev_pb(elev_cor, 0.4)
# elev_pb = elev_pb.astype(int)
elev_pb , elev_cor = cor_elev_pb(elev_cor, 0.2)
elev_pb = elev_pb.astype(int)
ploting.create_plotly_map(elev_pb, meta ,"Différence >0.4m", grids_hover=[elev_cor], info_hover=["elev"])


In [ ]:

A=[[1.,2.,3.],[4.,5.,6.], [7.,8.,9.]]
np.shape(A)
result = ndimage.generic_filter(A, np.nanmean, size=3, mode='constant', cval=np.nan)
mask = np.ones((3,3))
mask[1,1] = 0
result_mask = ndimage.generic_filter(A, np.nanmean,footprint=mask, mode='constant', cval=np.nan)
result, result_mask

In [ ]:
for i in [[]]:
    print("i")
